# Multi-Model Threshold Evaluation (FPR=20%)

This notebook recomputes thresholded metrics for six trained autoencoders using **mean** reconstruction error with an **FPR-controlled threshold** of **0.2**.

## Scope

This notebook is designed for your current setup:
- data available as ZIP files in Drive
- trained model checkpoints (`.pth`) available in Drive

Models included:
- ECNN-AE (optimized)
- CNN-AE Large
- CNN-AE Augmented
- CNN-AE
- ResNet-AE
- ResNet Finetuned (partial)

Outputs generated:
- Confusion matrix (counts) per model
- Normalized confusion matrix per model
- ROC curve per model
- Combined ROC comparison plot
- Consolidated metrics table (CSV + Markdown)

Threshold policy (fixed):
- Score method: `mean`
- Threshold method: `fpr`
- Parameter: `0.2`

In [ ]:
# Colab detection and Drive mount
import os

try:
    from google.colab import drive  # type: ignore
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
        print('Mounted Google Drive at /content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Running outside Colab; ensure Drive paths are accessible.')

In [ ]:
# Resolve repository and module paths
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/RifaDeen/symAD-ECNN.git'
CLONE_PATH = Path('/content/symAD-ECNN')

if IN_COLAB:
    if not CLONE_PATH.exists():
        print('Cloning repository...')
        subprocess.check_call(['git', 'clone', REPO_URL, str(CLONE_PATH)])
    REPO_ROOT = CLONE_PATH
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent:
        if (REPO_ROOT / 'README.md').exists() and (REPO_ROOT / 'notebooks').exists():
            break
        REPO_ROOT = REPO_ROOT.parent
    else:
        raise RuntimeError('Unable to locate project root. Open this notebook inside the project.')

EVALS_DIR = REPO_ROOT / 'notebooks' / 'evals'
MODULE_DIRS = [REPO_ROOT, EVALS_DIR, EVALS_DIR / 'ecnn_thresholding']
for p in MODULE_DIRS:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f'Repository root: {REPO_ROOT}')
print(f'Evals path: {EVALS_DIR}')

In [ ]:
# Imports
import os
import zipfile
import shutil
from glob import glob
from pathlib import Path
from typing import Dict, Tuple, List, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from config import DRIVE_PROJECT_ROOT, EVALUATIONS_ROOT, ensure_directories_exist
from metrics_utils import threshold_from_normal_scores, compute_full_metrics
from plotting_utils import (
    plot_confusion_matrix,
    plot_roc_curve,
    plot_multiple_roc_curves,
    save_figure,
 )
from io_utils import save_csv, save_markdown_table
from ecnn_model_loader import get_model_for_inference

ensure_directories_exist()
plt.style.use('seaborn-v0_8')
print('Imports and directories ready.')

In [ ]:
# Configuration
PROJECT_ROOT = DRIVE_PROJECT_ROOT if DRIVE_PROJECT_ROOT.exists() else REPO_ROOT
DATA_ROOT = PROJECT_ROOT / 'data'
MODELS_ROOT = PROJECT_ROOT / 'models' / 'saved_models'

TARGET_SCORE_METHOD = 'mean'
TARGET_FPR = 0.20
CLASS_NAMES = ['Normal', 'Anomaly']
FIGURE_SUBDIR = 'multi_model_fpr20'
TABLE_SUBDIR = 'ecnn_threshold_experiments'

# Optional manual overrides (set to exact zip path if auto-discovery picks wrong files)
NORMAL_ZIP_OVERRIDE = None
ANOMALY_ZIP_OVERRIDE = None

# Extraction root on local disk for faster IO
LOCAL_EXTRACT_ROOT = Path('/content/local_eval_data') if IN_COLAB else (REPO_ROOT / '.local_eval_data')

# Model config: checkpoint search + model type
MODEL_CONFIGS = [
    {
        'key': 'ecnn_opt',
        'display_name': 'ECNN-AE (Optimized)',
        'model_type': 'ecnn',
        'checkpoint_dirs': ['ecnn_optimized', '.'],
        'checkpoint_patterns': ['ecnn_optimized_best.pth', 'ecnn_best.pth', '*ecnn*optimized*best*.pth', '*ecnn*best*.pth'],
    },
    {
        'key': 'cnn_large',
        'display_name': 'CNN-AE Large',
        'model_type': 'cnn_large',
        'checkpoint_dirs': ['cnn_ae_large'],
        'checkpoint_patterns': ['cnn_large_best.pth', '*cnn*large*best*.pth'],
    },
    {
        'key': 'cnn_aug',
        'display_name': 'CNN-AE Augmented',
        'model_type': 'cnn_base',
        'checkpoint_dirs': ['cnn_ae_augmented'],
        'checkpoint_patterns': ['cnn_aug_best.pth', '*cnn*aug*best*.pth'],
    },
    {
        'key': 'cnn_base',
        'display_name': 'CNN-AE',
        'model_type': 'cnn_base',
        'checkpoint_dirs': ['cnn_ae'],
        'checkpoint_patterns': ['cnn_best.pth', '*cnn*best*.pth'],
    },
    {
        'key': 'resnet_ae',
        'display_name': 'ResNet-AE',
        'model_type': 'resnet_frozen',
        'checkpoint_dirs': ['resnet_ae'],
        'checkpoint_patterns': ['resnet_best.pth', '*resnet*best*.pth'],
    },
    {
        'key': 'resnet_ft',
        'display_name': 'ResNet Finetuned (partial)',
        'model_type': 'resnet_finetuned_partial',
        'checkpoint_dirs': ['resnet_finetuned_partial', 'resnet_finetuned_full', 'resnet_finetuned_none'],
        'checkpoint_patterns': ['resnet_best.pth', '*resnet*best*.pth'],
    },
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Models root: {MODELS_ROOT}")
print(f"Figure output: {EVALUATIONS_ROOT / 'figures' / FIGURE_SUBDIR}")
print(f"Table output: {EVALUATIONS_ROOT / 'tables' / TABLE_SUBDIR}")

In [ ]:
# Helper functions and model definitions
import torch.nn.functional as F
from PIL import Image

class CNNAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
        )
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 256, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 256)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        feats = self.encoder(x)
        b = feats.size(0)
        z = self.fc_encode(self.flatten(feats))
        dec = self.fc_decode(z).view(b, 256, 8, 8)
        return self.decoder(dec)

class LargeCNNAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),
        )
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 512, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 512)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 512, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        feats = self.encoder(x)
        b = feats.size(0)
        z = self.fc_encode(self.flatten(feats))
        dec = self.fc_decode(z).view(b, 512, 8, 8)
        return self.decoder(dec)

class ResNetAutoencoder(nn.Module):
    def __init__(self, finetune_strategy: str = 'none'):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])

        if finetune_strategy == 'none':
            for p in self.encoder.parameters():
                p.requires_grad = False
        elif finetune_strategy == 'partial':
            for i, module in enumerate(self.encoder.children()):
                train = i >= 7
                for p in module.parameters():
                    p.requires_grad = train
        elif finetune_strategy == 'full':
            for p in self.encoder.parameters():
                p.requires_grad = True

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 512, kernel_size=4, stride=1, padding=0),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

class SliceDataset(Dataset):
    def __init__(self, files: List[Path], mode: str = 'grayscale'):
        self.files = files
        self.mode = mode
        self.resnet_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    def __len__(self):
        return len(self.files)

    def _load_slice(self, path: Path) -> np.ndarray:
        if path.suffix.lower() == '.npy':
            arr = np.load(path)
        else:
            arr = np.array(Image.open(path).convert('L'))
            arr = arr.astype(np.float32) / 255.0
        arr = arr.astype(np.float32)
        if arr.max() > 1.0:
            arr = arr / 255.0
        if arr.ndim == 3:
            arr = arr.squeeze()
        return arr

    def __getitem__(self, idx):
        arr = self._load_slice(self.files[idx])
        gray = torch.from_numpy(arr).float().unsqueeze(0)
        if gray.shape[-2:] != (128, 128):
            gray = F.interpolate(gray.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False).squeeze(0)

        if self.mode == 'resnet':
            img_uint8 = (gray.squeeze(0).numpy() * 255.0).clip(0, 255).astype(np.uint8)
            rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            inp = self.resnet_transform(rgb)
            tgt = gray
            return inp, tgt

        return gray, gray

def find_files(root_dir: Path) -> List[Path]:
    exts = ['*.npy', '*.png', '*.jpg', '*.jpeg']
    files = []
    for ext in exts:
        files.extend(root_dir.rglob(ext))
    return sorted([p for p in files if p.is_file()])

def discover_zip(zip_override: Optional[str], target: str) -> Path:
    if zip_override is not None:
        z = Path(zip_override)
        if not z.exists():
            raise FileNotFoundError(f'Override zip not found: {z}')
        return z

    all_zips = sorted(DATA_ROOT.rglob('*.zip'))
    if not all_zips:
        raise FileNotFoundError(f'No zip files found under {DATA_ROOT}')

    def score_zip(p: Path) -> int:
        n = p.name.lower()
        s = 0
        if target == 'normal':
            if 'val' in n: s += 5
            if 'ixi' in n: s += 5
            if 'normal' in n: s += 4
            if 'train' in n: s += 1
            if 'brats' in n: s -= 10
        else:
            if 'brats' in n: s += 8
            if 'anomaly' in n or 'tumor' in n: s += 6
            if 'test' in n: s += 3
            if 'ixi' in n: s -= 6
            if 'val' in n: s -= 3
        if 'fast' in n: s += 1
        return s

    ranked = sorted(all_zips, key=lambda p: (score_zip(p), p.name.lower()), reverse=True)
    best = ranked[0]
    print(f"Selected {target} zip: {best}")
    return best

def extract_zip(zip_path: Path, out_dir: Path) -> Path:
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)
    return out_dir

def resolve_checkpoint_path(cfg: Dict) -> Path:
    roots = []
    for sub in cfg['checkpoint_dirs']:
        if sub == '.':
            roots.append(MODELS_ROOT)
        else:
            roots.append(MODELS_ROOT / sub)

    for root in roots:
        if not root.exists():
            continue
        for pattern in cfg['checkpoint_patterns']:
            matches = sorted(root.rglob(pattern))
            if matches:
                chosen = matches[0]
                print(f"{cfg['display_name']} checkpoint: {chosen}")
                return chosen

    raise FileNotFoundError(f"No checkpoint found for {cfg['display_name']} under {MODELS_ROOT}")

def get_state_dict(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        if 'model_state_dict' in ckpt_obj and isinstance(ckpt_obj['model_state_dict'], dict):
            return ckpt_obj['model_state_dict']
        if 'state_dict' in ckpt_obj and isinstance(ckpt_obj['state_dict'], dict):
            return ckpt_obj['state_dict']
    return ckpt_obj

def load_model_for_config(cfg: Dict, device: str):
    ckpt_path = resolve_checkpoint_path(cfg)

    if cfg['model_type'] == 'ecnn':
        model, info = get_model_for_inference(ckpt_path, device)
        return model, ckpt_path

    checkpoint = torch.load(ckpt_path, map_location=device)
    state_dict = get_state_dict(checkpoint)

    if cfg['model_type'] == 'cnn_large':
        latent_dim = int(state_dict['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in state_dict else 512
        model = LargeCNNAutoencoder(latent_dim=latent_dim)
    elif cfg['model_type'] == 'cnn_base':
        latent_dim = int(state_dict['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in state_dict else 256
        model = CNNAutoencoder(latent_dim=latent_dim)
    elif cfg['model_type'] == 'resnet_frozen':
        model = ResNetAutoencoder(finetune_strategy='none')
    elif cfg['model_type'] == 'resnet_finetuned_partial':
        model = ResNetAutoencoder(finetune_strategy='partial')
    else:
        raise ValueError(f"Unsupported model type: {cfg['model_type']}")

    model.load_state_dict(state_dict, strict=False)
    model = model.to(device)
    model.eval()
    return model, ckpt_path

@torch.no_grad()
def compute_errors(model: nn.Module, dataloader: DataLoader, device: str) -> np.ndarray:
    model.eval()
    errors = []
    for x, y in tqdm(dataloader, desc='Computing reconstruction errors', leave=False):
        x = x.to(device)
        y = y.to(device)
        recon = model(x)
        mse = F.mse_loss(recon, y, reduction='none').view(x.size(0), -1).mean(dim=1)
        errors.extend(mse.detach().cpu().numpy().tolist())
    return np.asarray(errors, dtype=np.float32)

def summarize_scores(scores: np.ndarray) -> Dict[str, float]:
    return {
        'count': int(scores.size),
        'mean': float(np.mean(scores)),
        'std': float(np.std(scores)),
        'min': float(np.min(scores)),
        'max': float(np.max(scores)),
    }

print('Helper functions ready.')

In [ ]:
# Prepare data from zip files and evaluate all models
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

normal_zip = discover_zip(NORMAL_ZIP_OVERRIDE, target='normal')
anomaly_zip = discover_zip(ANOMALY_ZIP_OVERRIDE, target='anomaly')

normal_dir = extract_zip(normal_zip, LOCAL_EXTRACT_ROOT / 'normal')
anomaly_dir = extract_zip(anomaly_zip, LOCAL_EXTRACT_ROOT / 'anomaly')

normal_files = find_files(normal_dir)
anomaly_files = find_files(anomaly_dir)

print(f'Normal files: {len(normal_files)} from {normal_dir}')
print(f'Anomaly files: {len(anomaly_files)} from {anomaly_dir}')
if len(normal_files) == 0 or len(anomaly_files) == 0:
    raise RuntimeError('No data files found after extraction. Verify zip contents.')

model_summaries = []
roc_payload = []
failed_models = []

for cfg in MODEL_CONFIGS:
    print(f"\n=== {cfg['display_name']} ===")
    try:
        model, checkpoint_path = load_model_for_config(cfg, device)
        mode = 'resnet' if 'resnet' in cfg['model_type'] else 'grayscale'

        normal_ds = SliceDataset(normal_files, mode=mode)
        anomaly_ds = SliceDataset(anomaly_files, mode=mode)

        normal_loader = DataLoader(normal_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
        anomaly_loader = DataLoader(anomaly_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

        normal_scores = compute_errors(model, normal_loader, device)
        anomaly_scores = compute_errors(model, anomaly_loader, device)

        threshold = threshold_from_normal_scores(normal_scores, target_fpr=TARGET_FPR)
        y_true = np.concatenate([np.zeros(len(normal_scores), dtype=int), np.ones(len(anomaly_scores), dtype=int)])
        y_scores = np.concatenate([normal_scores, anomaly_scores])
        metrics = compute_full_metrics(y_true, y_scores, threshold)
        metrics['threshold'] = float(threshold)

        y_pred = (y_scores >= threshold).astype(int)

        fig_counts, ax_counts = plot_confusion_matrix(
            y_true, y_pred, class_names=CLASS_NAMES,
            title=f"{cfg['display_name']} - Confusion Matrix (Counts)",
            normalize=False,
        )
        save_figure(fig_counts, f"{cfg['key']}_confusion_counts_fpr20", subdir=FIGURE_SUBDIR)

        fig_norm, ax_norm = plot_confusion_matrix(
            y_true, y_pred, class_names=CLASS_NAMES,
            title=f"{cfg['display_name']} - Confusion Matrix (Normalized)",
            normalize=True,
        )
        save_figure(fig_norm, f"{cfg['key']}_confusion_normalized_fpr20", subdir=FIGURE_SUBDIR)

        fig_roc, ax_roc = plot_roc_curve(
            y_true, y_scores, auroc=metrics['auroc'],
            title=f"{cfg['display_name']} - ROC Curve",
            label=f"{cfg['display_name']} (AUROC={metrics['auroc']:.4f})",
        )
        save_figure(fig_roc, f"{cfg['key']}_roc_fpr20", subdir=FIGURE_SUBDIR)

        model_summaries.append({
            'model_key': cfg['key'],
            'model_name': cfg['display_name'],
            'checkpoint_path': str(checkpoint_path),
            'score_method': TARGET_SCORE_METHOD,
            'threshold_method': 'fpr',
            'target_fpr': TARGET_FPR,
            'threshold': metrics['threshold'],
            'accuracy': metrics['accuracy'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'specificity': metrics['specificity'],
            'f1_score': metrics['f1_score'],
            'fpr': metrics['fpr'],
            'fnr': metrics['fnr'],
            'auroc': metrics['auroc'],
            'auprc': metrics['auprc'],
            'tp': metrics['tp'],
            'tn': metrics['tn'],
            'fp': metrics['fp'],
            'fn': metrics['fn'],
            'normal_count': int(len(normal_scores)),
            'anomaly_count': int(len(anomaly_scores)),
            'normal_mean': float(normal_scores.mean()),
            'normal_std': float(normal_scores.std()),
            'anomaly_mean': float(anomaly_scores.mean()),
            'anomaly_std': float(anomaly_scores.std()),
        })

        roc_payload.append({
            'y_true': y_true,
            'y_scores': y_scores,
            'label': cfg['display_name'],
            'auroc': metrics['auroc'],
        })

        # Persist raw error arrays for reuse
        model_result_dir = EVALUATIONS_ROOT / 'json' / 'multi_model_fpr20_errors' / cfg['key']
        model_result_dir.mkdir(parents=True, exist_ok=True)
        np.save(model_result_dir / 'normal_errors.npy', normal_scores)
        np.save(model_result_dir / 'anomaly_errors.npy', anomaly_scores)

    except Exception as exc:
        print(f"WARNING: Skipping {cfg['display_name']} -> {exc}")
        failed_models.append({'model': cfg['display_name'], 'reason': str(exc)})

if failed_models:
    print('\nSkipped models:')
    for item in failed_models:
        print(f" - {item['model']}: {item['reason']}")
else:
    print('\nAll requested models evaluated successfully.')

In [ ]:
# Metrics table display and export
from IPython.display import display

if model_summaries:
    metrics_df = pd.DataFrame(model_summaries)
    metrics_df = metrics_df.sort_values('auroc', ascending=False).reset_index(drop=True)

    display(
        metrics_df.style.format({
            'target_fpr': '{:.2f}',
            'threshold': '{:.6f}',
            'accuracy': '{:.4f}',
            'precision': '{:.4f}',
            'recall': '{:.4f}',
            'specificity': '{:.4f}',
            'f1_score': '{:.4f}',
            'fpr': '{:.4f}',
            'fnr': '{:.4f}',
            'auroc': '{:.4f}',
            'auprc': '{:.4f}',
            'normal_mean': '{:.6f}',
            'normal_std': '{:.6f}',
            'anomaly_mean': '{:.6f}',
            'anomaly_std': '{:.6f}',
        })
    )

    csv_path = save_csv(metrics_df, 'multi_model_fpr20_metrics.csv', subdir=TABLE_SUBDIR)
    md_path = save_markdown_table(
        metrics_df,
        'multi_model_fpr20_metrics.md',
        title='Multi-Model Metrics (Score=mean, Threshold=fpr, Param=0.2)',
        subdir=TABLE_SUBDIR,
    )

    print(f'CSV saved: {csv_path}')
    print(f'Markdown saved: {md_path}')
else:
    print('No successful evaluations. Verify score arrays exist for all models.')

In [ ]:
# Combined ROC plot
if roc_payload:
    fig, ax = plot_multiple_roc_curves(
        roc_payload,
        title='ROC Comparison (All Models, Score=mean, Threshold=fpr@0.2)',
    )
    save_figure(fig, 'multi_model_fpr20_roc_comparison', subdir=FIGURE_SUBDIR)
    print('Saved combined ROC figure.')
else:
    print('No ROC payload available.')

## Notes

- This notebook works directly from ZIP data + checkpoint `.pth` files on Drive.
- If auto-selected ZIP files are incorrect, set `NORMAL_ZIP_OVERRIDE` and/or `ANOMALY_ZIP_OVERRIDE` in the configuration cell.
- Extracted files are cached under `LOCAL_EXTRACT_ROOT` for faster evaluation.
- Raw per-model error arrays are saved to `evaluations/json/multi_model_fpr20_errors/<model_key>/`.
- Thresholding policy is fixed to: **Score = mean**, **Threshold = fpr**, **Param = 0.2**.